# GPU Baseline — Llama-3.2-3B-Instruct (BF16)

This notebook benchmarks the **unquantized** Llama-3.2-3B-Instruct model on GPU, providing a baseline for comparing against the quantized (GGUF) variants running on the Pixel 6a.

**Runtime:** Use **T4 GPU** (free Colab tier). A100/V100 will be faster.

**Expected time:** ~15 min total (model download + 20 trials × 2 context sizes)

**What we measure:** Decode throughput (tokens/sec) matching the same context sizes and prompts as the on-device llama.cpp benchmark.

## Step 1: Install Dependencies

In [ ]:
!pip install -q transformers accelerate torch huggingface_hub

## Step 2: Authenticate with HuggingFace

Llama-3.2 is a gated model. You need to:
1. Accept the license at https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct
2. Create a token at https://huggingface.co/settings/tokens
3. Paste it below when prompted

In [ ]:
from huggingface_hub import login
login()  # Enter your HuggingFace token when prompted
# Alternative: set HF_TOKEN env var and skip this cell

## Step 3: Check GPU

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
    print(f"BF16 supported: {torch.cuda.is_bf16_supported()}")
else:
    print("WARNING: No GPU — go to Runtime > Change runtime type > T4 GPU")

## Step 4: Run the Benchmark

This downloads Llama-3.2-3B-Instruct (~6.4 GB) and runs 10 trials each for ctx=256 and ctx=1024.

In [ ]:
# Option A: Clone the repo (if running from GitHub)
# !git clone https://github.com/KrisDcosta/edge-llm-bench.git
# %cd edge-llm-bench

# Option B: Upload gpu_baseline.py manually and run it
# Upload scripts/gpu_baseline.py using the Files panel on the left

# Option C: Run inline (cell below)

In [ ]:
# Run the benchmark (if gpu_baseline.py is uploaded to Colab)
!python scripts/gpu_baseline.py \
    --model meta-llama/Llama-3.2-3B-Instruct \
    --output results/gpu_baseline_colab.json \
    --ctx-sizes 256 1024

### OR: Run Inline (no file upload needed)

In [ ]:
import json, os, time, statistics
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
OUTPUT_TOKENS = 128
NUM_TRIALS = 10
WARMUP_TRIALS = 2

PROMPTS = [
    "What is the capital of France? Answer in one word.",
    "Explain the difference between machine learning and deep learning in two sentences.",
    "What is the Pythagorean theorem? Give the formula and a brief explanation.",
    "Write a short paragraph describing the water cycle, including evaporation, condensation, and precipitation.",
    "List three advantages and three disadvantages of renewable energy sources compared to fossil fuels.",
]

# Load model
print("Loading model...")
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype, device_map="auto")
model.eval()
print(f"Model loaded. dtype={dtype}, device={next(model.parameters()).device}")

results = {"model": MODEL_ID, "dtype": str(dtype), "benchmarks": {}}

for ctx_str in ["256", "1024"]:
    ctx_label = f"ctx_{ctx_str}"
    print(f"\n--- Benchmarking {ctx_label} ---")
    all_tps = []

    for i in range(WARMUP_TRIALS + NUM_TRIALS):
        is_warmup = i < WARMUP_TRIALS
        prompt = PROMPTS[i % len(PROMPTS)]
        inputs = tokenizer(prompt, return_tensors="pt").to(next(model.parameters()).device)
        in_tokens = inputs["input_ids"].shape[1]

        if torch.cuda.is_available(): torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=OUTPUT_TOKENS, do_sample=False,
                                 pad_token_id=tokenizer.eos_token_id)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        elapsed = time.perf_counter() - t0

        out_tokens = out.shape[1] - in_tokens
        tps = out_tokens / elapsed
        label = "WARMUP" if is_warmup else f"T{i-WARMUP_TRIALS+1:02d}"
        print(f"  {label}: {out_tokens} tokens | {tps:.1f} tok/s | {elapsed:.2f}s")
        if not is_warmup:
            all_tps.append(tps)

    mean_tps = statistics.mean(all_tps)
    std_tps = statistics.stdev(all_tps)
    print(f"  SUMMARY: {mean_tps:.2f} ± {std_tps:.2f} tok/s [{min(all_tps):.2f}–{max(all_tps):.2f}]")
    results["benchmarks"][ctx_label] = {
        "mean_decode_tps": round(mean_tps, 3),
        "std_decode_tps": round(std_tps, 3),
        "min_decode_tps": round(min(all_tps), 3),
        "max_decode_tps": round(max(all_tps), 3),
    }

# Save results
os.makedirs("results", exist_ok=True)
with open("results/gpu_baseline_colab.json", "w") as f:
    json.dump(results, f, indent=2)

print("\nResults saved to results/gpu_baseline_colab.json")
print(json.dumps(results["benchmarks"], indent=2))

## Step 5: Compare to Pixel 6a On-Device Results

In [ ]:
# On-device TPS from run-20260305T174113.jsonl (best benchmark run)
# ctx=256 mean decode TPS per variant:
device_tps = {
    "Q2_K":   {"ctx_256": 5.66, "ctx_1024": 5.54},
    "Q3_K_M": {"ctx_256": 5.44, "ctx_1024": 5.31},
    "Q4_K_M": {"ctx_256": 5.32, "ctx_1024": 5.20},
    "Q6_K":   {"ctx_256": 4.88, "ctx_1024": 4.76},
    "Q8_0":   {"ctx_256": 4.41, "ctx_1024": 4.29},
    "F16":    {"ctx_256": 2.11, "ctx_1024": 2.05},
}

# Load GPU results
with open("results/gpu_baseline_colab.json") as f:
    gpu_results = json.load(f)

gpu_ctx256 = gpu_results["benchmarks"]["ctx_256"]["mean_decode_tps"]
gpu_ctx1024 = gpu_results["benchmarks"]["ctx_1024"]["mean_decode_tps"]

print(f"GPU ({gpu_results.get('dtype','BF16')}):")
print(f"  ctx=256:  {gpu_ctx256:.1f} tok/s")
print(f"  ctx=1024: {gpu_ctx1024:.1f} tok/s")
print()
print("Speedup vs Pixel 6a (ctx=256):")
print(f"  {'Variant':<10} {'Device':>10} {'GPU':>10} {'Speedup':>10}")
print("-" * 44)
for variant, tps in device_tps.items():
    speedup = gpu_ctx256 / tps["ctx_256"]
    print(f"  {variant:<10} {tps['ctx_256']:>10.2f} {gpu_ctx256:>10.1f} {speedup:>9.1f}x")

## Step 6: Download Results

Download `results/gpu_baseline_colab.json` from the Files panel (left sidebar) and add it to the project repo under `results/`.

In [ ]:
# Download the results file
from google.colab import files
files.download("results/gpu_baseline_colab.json")